Credit Card Fraud Detection

The dataset contains transactions made by credit cards in September 2013 by European cardholders.
This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.

It contains only numerical input variables which are the result of a PCA transformation. Unfortunately, due to confidentiality issues, we cannot provide the original features and more background information about the data. Features V1, V2, … V28 are the principal components obtained with PCA, the only features which have not been transformed with PCA are 'Time' and 'Amount'. Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset. The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-sensitive learning. Feature 'Class' is the response variable and it takes value 1 in case of fraud and 0 otherwise.

Given the class imbalance ratio, we recommend measuring the accuracy using the Area Under the Precision-Recall Curve (AUPRC). Confusion matrix accuracy is not meaningful for unbalanced classification.



Obtener datos del dataset

In [1]:
import kagglehub
import pandas as pd
import mlflow
import mlflow.sklearn
from prefect import flow, task
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score
from imblearn.under_sampling import RandomUnderSampler


@task(retries=2, retry_delay_seconds=60)
def download_data():
    """Descarga el dataset desde Kaggle usando kagglehub."""
    print("Descargando dataset...")
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    # El path suele ser una carpeta, buscamos el .csv
    import os

    for file in os.listdir(path):
        if file.endswith(".csv"):
            return os.path.join(path, file)


@task
def prepare_data(file_path):
    """Carga y prepara los datos para el entrenamiento."""
    df = pd.read_csv(file_path)
    X = df.drop(["Class", "Time"], axis=1)  # 'Time' suele no ser útil sin ingeniería
    y = df["Class"]

    # Manejo de desbalance: Undersampling para balancear rápido el experimento
    rus = RandomUnderSampler(random_state=42)
    X_res, y_res = rus.fit_resample(X, y)

    return train_test_split(X_res, y_res, test_size=0.2, random_state=42)


@task
def train_and_log_model(X_train, X_test, y_train, y_test):
    """Entrena el modelo y registra métricas/artefactos en MLflow."""
    with mlflow.start_run(run_name="RandomForest_UnderSample"):
        n_estimators = 100
        model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        model.fit(X_train, y_train)

        # Predicciones
        y_pred = model.predict(X_test)
        auprc = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])

        # Log de parámetros y métricas
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_metric("auprc", auprc)

        # Log del modelo
        mlflow.sklearn.log_model(model, "fraud-model")

        print(f"Modelo entrenado con AUPRC: {auprc:.4f}")
        return mlflow.active_run().info.run_id


@flow(name="Credit Card Fraud MLOps Pipeline")
def fraud_detection_pipeline():
    # 1. Ingesta
    csv_path = download_data()

    # 2. Preprocesamiento
    X_train, X_test, y_train, y_test = prepare_data(csv_path)

    # 3. Entrenamiento y Tracking
    run_id = train_and_log_model(X_train, X_test, y_train, y_test)

    print(f"Pipeline completado. Run ID: {run_id}")


if __name__ == "__main__":
    fraud_detection_pipeline()

/Users/diegofernandonunezdiaz/Library/Mobile Documents/com~apple~CloudDocs/CURSOS/UMEDELLIN/nube/proyecto_ml/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


19:02:15.237 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8245
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

19:02:25.789 | INFO    | Flow run 'industrious-dugong' - Beginning flow run 'industrious-dugong' for flow 'Credit Card Fraud MLOps Pipeline'

Descargando dataset...


19:02:28.926 | INFO    | Task run 'download_data-742' - Finished in state Completed()

...

2026/04/21 19:02:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/21 19:02:33 INFO mlflow.store.db.utils: Updating database tables
2026/04/21 19:02:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 19:02:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/21 19:02:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Modelo entrenado con AUPRC: 0.9797


19:02:46.195 | INFO    | Task run 'train_and_log_model-41a' - Finished in state Completed()

Pipeline completado. Run ID: 924e8549cc75423ea5bca82e5da14789


19:02:47.161 | INFO    | Flow run 'industrious-dugong' - Finished in state Completed()